In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install numpy==2.1.3

In [ ]:
!pip install pandas==2.2.3

In [ ]:
!pip install matplotlib==3.9.2

In [ ]:
!pip install seaborn==0.13.2

In [ ]:
!pip install scipy==1.15.2

In [ ]:
!pip install statsmodels==0.14.4

In [ ]:
!pip install tabulate==0.9.0

In [ ]:
!pip install psycopg2-binary

In [ ]:
!pip install sqlalchemy

In [ ]:
!pip install natsort

In [ ]:
!pip install python-dotenv

In [ ]:
!pip freeze

In [ ]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

In [ ]:
project_id: int = 0 # @TODO: Set a valid project ID.

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env", override=True)

db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "postgres")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "postgres")

connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
conn = create_engine(connection_string)

## Number of test in-/exclusions

In [ ]:
query = f"SELECT project_id, count(DISTINCT test_class_qualified_name) AS test_class_count, count(*) AS test_method_count, sum(is_included::int) AS included_count FROM test WHERE project_id = {project_id} GROUP BY project_id"
df = pd.read_sql_query(query, conn)
df['excluded_count'] = df['test_method_count'] - df['included_count']
df

## Number of assertion in-/exclusions

In [ ]:
query = f"SELECT project_id, count(DISTINCT test_id) test_count, count(*) AS assertion_count, sum(is_included::int) AS included_count FROM assertion WHERE project_id = {project_id} GROUP BY project_id"
df = pd.read_sql_query(query, conn)
df['excluded_count'] = df['assertion_count'] - df['included_count']
df

The number of tests is usually lower here than above due to tests that do not contain any assertions. 

## Number of generalization in-/exclusions

In [ ]:
query = f"SELECT project_id, variant, count(*) AS total_count, sum(is_included::int) AS included_count FROM generalization WHERE project_id = {project_id} GROUP BY project_id, variant"
df = pd.read_sql_query(query, conn)
df['excluded_count'] = df['total_count'] - df['included_count']
df

## Number of test / assertion / generalization exclusions per task

In [ ]:
query = f"""
SELECT project_id, 1 AS type_id, 'TEST' AS item_type, '-' AS variant, exclusion_info FROM test WHERE project_id = {project_id} AND is_included = false
UNION ALL
SELECT project_id, 2, 'ASSERTION', '-', exclusion_info FROM assertion WHERE project_id = {project_id} AND is_included = false
UNION ALL
SELECT project_id, 3, 'GENERALIZATION', variant, exclusion_info FROM generalization WHERE project_id = {project_id} AND is_included = false
"""
df = pd.read_sql_query(query, conn)

def extract_task_name(s):
    match = re.search(r'\b(\w+){', s)
    return match.group(1) if match else None

df['exclusion_info'] = df['exclusion_info'].apply(extract_task_name)

df = df.pivot_table(index=['project_id', 'type_id', 'item_type', 'variant'], columns='exclusion_info', aggfunc='size', fill_value=0)
df['Total Exclusions'] = df.sum(axis=1)
df = df[['Total Exclusions'] + [col for col in df if col != 'Total Exclusions']]

df

## Causes of task failure-based exclusions

In [ ]:
query = f"SELECT project_id, step, stage, variant, info FROM task WHERE project_id = {project_id} AND status = 'FAILED'"
df = pd.read_sql_query(query, conn, params={"project_id": project_id})

failure_types = [
    'code too large',
    'Depth limit of 100 exceeded',
    'Unable to identify test report path',
    'PC size limit exceeded',
    'Execution timeout exceeded',
    'No assertions found',
    'might be inherited',
    'contains \\(static\\) initializers',
    'Failed to identify valid type for parameter',
    'has no @Test annotation',
    'AssertionFailedError',
    'java.lang.ArithmeticException: !!!div by 0',
    'java.lang.ClassNotFoundException: class not found: java.lang.NoSuchMethodException!!',
    'method arguments do not match with JPF\'s symbolic.method configuration',
    'ATHROW cannot be cast to gov.nasa.jpf.jvm.bytecode.JVMReturnInstruction',
    'INVOKESTATIC cannot be cast to gov.nasa.jpf.jvm.bytecode.JVMReturnInstruction',
    'java.lang.NullPointerException',
    'java.lang.OutOfMemoryError: Java heap space',
    'java.lang.OutOfMemoryError: GC overhead limit exceeded',
    'NEWARRAY: symbolic array length',
    'java.lang.ClassCastException',
    'Unable to transform operation',
    'java.io.FileNotFoundException',
    'Failed to collect input/output specification for unknown reason',
    'exception during stateAdvanced',
    'exception during searchConstraintHit',
    'exception during methodExited',
    'AssertionError',
    'no peer'
]

def categorize_failure(info):
    if pd.isna(info):
        return '<No Info>'

    for failure_type in failure_types:
        if failure_type in info:
            return failure_type
    return '<Other>'

# Apply categorization
df['failure_type'] = df['info'].apply(categorize_failure)

# Count failures by type
failure_counts = df['failure_type'].value_counts().reset_index()
failure_counts.columns = ['failure_type', 'count']

# Get stages for each failure type
stage_info = df.groupby('failure_type')['stage'].agg(lambda x: ', '.join(sorted(set(x)))).reset_index()

# Merge the stage information with the failure counts
failure_summary = pd.merge(failure_counts, stage_info, on='failure_type', how='left')

# Add percentage column
failure_summary['percentage'] = failure_summary.apply(
    lambda row: row['count'] / len(df) * 100, 
    axis=1
)

# Reorder columns to put stage at the end
failure_summary = failure_summary[['failure_type', 'count', 'percentage', 'stage']]

# Add total row
total_count = len(df)
total_stages = ', '.join(sorted(df['stage'].unique()))
total_row = pd.DataFrame({
    'failure_type': ['<Total>'], 
    'count': [total_count], 
    'percentage': [100.0],
    'stage': [total_stages]
})
failure_summary = pd.concat([failure_summary, total_row], ignore_index=True)

# Sort by count in descending order
failure_summary = failure_summary.sort_values(by='count', ascending=False).reset_index(drop=True)

# List failures that don't match known failure types
other_failures = df[df['failure_type'] == '<Other>']
if len(other_failures) > 0:
    print(f"\nUnmatched failure messages ({len(other_failures)}):")
    for info in other_failures['info'].head(10).values:
        print(f"\n{info[:200]}..." if len(str(info)) > 200 else f"\n{info}")

    if len(other_failures) > 10:
        print(f"\n... and {len(other_failures) - 10} more")

# Display the results
failure_summary

## Causes of filtering-based exclusions

In [ ]:
from sqlalchemy import text

def process_info(info):
    lines = info.split('\n')
    result = {}
    for line in lines:
        # Split on first colon only
        split_idx = line.find(': ')
        if split_idx != -1:
            key = line[:split_idx]
            value = line[split_idx + 2:]  # +2 to skip ': '
            result[key] = 1 if value.startswith('REJECT') else 0
    return result

def get_exclusions(conn, project_id, table_name):
    # Use SQLAlchemy's text() for proper parameter handling
    query = text(f"""
        SELECT project_id, exclusion_info, '{table_name}' as source 
        FROM {table_name} 
        WHERE project_id = :project_id 
        AND exclusion_info LIKE '%TestFilteringTask%'
    """)
    # Pass parameters as a dictionary
    params = {"project_id": project_id}
    return pd.read_sql_query(query, conn, params=params)

tables = ['test', 'assertion', 'generalization']
dfs = [get_exclusions(conn, project_id, table) for table in tables]

combined_df = pd.concat(dfs, ignore_index=True)
combined_df['exclusion_info_dict'] = combined_df['exclusion_info'].apply(process_info)

filter_df = pd.json_normalize(combined_df['exclusion_info_dict'])
filter_df = filter_df.fillna(0).astype(int)

combined_df = pd.concat([combined_df, filter_df], axis=1)
combined_df = combined_df.drop(columns=['exclusion_info', 'exclusion_info_dict'])

result = combined_df.groupby(['project_id', 'source']).sum()
result


## Mutation testing results pre-/post-generalization

In [ ]:
query = f"SELECT project_id, step, stage, variant, status, is_detected FROM pit_mutation_report WHERE project_id = {project_id}"
df = pd.read_sql_query(query, conn)
result = pd.DataFrame()

if not df.empty:
    df['variant'] = df['variant'].fillna('ORIGINAL')

    mutation_status_categories = ['SURVIVED', 'KILLED', 'TIMED_OUT', 'NO_COVERAGE', 'NON_VIABLE', 'MEMORY_ERROR', 'RUN_ERROR']
    mutation_status_categories = [c for c in mutation_status_categories if c in df['status'].unique()]
    df['status'] = pd.Categorical(df['status'], categories=mutation_status_categories)

    result = pd.pivot_table(
        df,
        index=['project_id', 'step', 'stage', 'variant'],
        columns='status',
        values='status',
        aggfunc='count',
        fill_value=0,
        observed=True
    )

    result['TOTAL'] = result[mutation_status_categories].sum(axis=1)
    result = result[['TOTAL'] + mutation_status_categories]

    result['DETECTED'] = pd.pivot_table(
        df,
        index=['project_id', 'step', 'stage', 'variant'],
        values='is_detected',
        aggfunc='sum',
        fill_value=0
    )

    # Calculate detected mutations safely
    killed = result['KILLED'] if 'KILLED' in result.columns else 0
    timed_out = result['TIMED_OUT'] if 'TIMED_OUT' in result.columns else 0
    no_coverage = result['NO_COVERAGE'] if 'NO_COVERAGE' in result.columns else 0

    # Calculate percentage of detected mutations
    result['% detected'] = result['DETECTED'] / result['TOTAL'].replace(0, 1)  # Avoid division by zero

    # Calculate percentage detected of covered mutations
    covered_mutations = result['TOTAL'] - no_coverage
    detected_mutations = killed + timed_out
    result['% detected of covered'] = detected_mutations / covered_mutations.replace(0, 1)  # Avoid division by zero

    result.reset_index(inplace=True)

result

## Analysis of mutation testing results

In [ ]:
# 1. Mutation Kill Rate - Overall and by variant
query = f"""
SELECT 
    CASE WHEN variant IS NULL THEN 'ORIGINAL' ELSE variant END AS variant,
    COUNT(*) AS total_mutations,
    SUM(is_detected::int) AS killed_mutations,
    ROUND(SUM(is_detected::int) * 100.0 / COUNT(*), 2) AS kill_rate
FROM pit_mutation_report
WHERE project_id = {project_id}
GROUP BY variant
ORDER BY kill_rate DESC
"""
kill_rate_df = pd.read_sql_query(query, conn)
print("Mutation Kill Rate by Variant:")
display(kill_rate_df)

# Visualization for Kill Rates
plt.figure(figsize=(10, 6))
bars = plt.bar(kill_rate_df['variant'], kill_rate_df['kill_rate'], color='skyblue')
plt.xlabel('Variant')
plt.ylabel('Kill Rate (%)')
plt.title('Mutation Kill Rate by Variant')
plt.ylim(0, 100)

# Add the values on top of the bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{height:.2f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# 2. Impact of Mutator Types
query = f"""
SELECT 
    CASE WHEN variant IS NULL THEN 'ORIGINAL' ELSE variant END AS variant,
    mutator,
    COUNT(*) AS total_mutations,
    SUM(is_detected::int) AS killed_mutations,
    ROUND(SUM(is_detected::int) * 100.0 / COUNT(*), 2) AS kill_rate
FROM pit_mutation_report
WHERE project_id = {project_id}
GROUP BY variant, mutator
ORDER BY variant, kill_rate DESC
"""
mutator_df = pd.read_sql_query(query, conn)

# Extract just the class name from the fully qualified mutator path
mutator_df['mutator_name'] = mutator_df['mutator'].apply(lambda x: x.split('.')[-1])

# Create a pivot table to compare mutators across variants
pivot_df = mutator_df.pivot_table(
    index='mutator_name',
    columns='variant',
    values=['total_mutations', 'killed_mutations', 'kill_rate'],
    aggfunc='first'  # Use the first value since they should be the same across variants
)

# Check if total_mutations are the same across variants
total_mutations_df = mutator_df.pivot_table(
    index='mutator_name',
    columns='variant',
    values='total_mutations'
)

# Determine if total_mutations are consistent across variants
consistent_mutations = total_mutations_df.eq(total_mutations_df.iloc[:, 0], axis=0).all(axis=1)
inconsistent_mutators = consistent_mutations[~consistent_mutations].index.tolist()

if inconsistent_mutators:
    print("Note: The following mutators have different total_mutations counts across variants:")
    for mutator in inconsistent_mutators:
        print(f"- {mutator}")
    print()

# Flatten the column multi-index for better readability
pivot_df.columns = [f"{col[0]}_{col[1]}" for col in pivot_df.columns]

# Sort by total mutations (using the first variant's count)
first_variant = mutator_df['variant'].unique()[0]
total_col = f"total_mutations_{first_variant}"
if total_col in pivot_df.columns:
    pivot_df = pivot_df.sort_values(by=total_col, ascending=False)

print("\nMutator Performance Across All Variants:")
display(pivot_df)

# Visualization for top 10 mutators by frequency
top_mutators = pivot_df.head(10)

# Create a grouped bar chart for kill rates across variants
plt.figure(figsize=(14, 8))
bar_width = 0.8 / len(mutator_df['variant'].unique())
x = np.arange(len(top_mutators))

for i, variant in enumerate(mutator_df['variant'].unique()):
    kill_rate_col = f"kill_rate_{variant}"
    if kill_rate_col in top_mutators.columns:
        plt.bar(x + i * bar_width, 
                top_mutators[kill_rate_col], 
                width=bar_width, 
                label=variant)

plt.xlabel('Mutator')
plt.ylabel('Kill Rate (%)')
plt.title('Top 10 Mutators by Frequency - Kill Rate Comparison Across Variants')
plt.xticks(x + bar_width * (len(mutator_df['variant'].unique()) - 1) / 2, top_mutators.index, rotation=45, ha='right')
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()

# 3. Improvement Analysis - Comparing variants to ORIGINAL
print("\nImprovement Analysis - Comparing variants to ORIGINAL:")

# First, ensure we have the ORIGINAL variant in our data
if 'ORIGINAL' in mutator_df['variant'].unique():
    # Create a pivot table focused on kill rates
    kill_rate_pivot = mutator_df.pivot_table(
        index='mutator_name',
        columns='variant',
        values='kill_rate'
    )

    # Calculate improvement over ORIGINAL for each variant
    improvement_df = pd.DataFrame(index=kill_rate_pivot.index)

    for variant in kill_rate_pivot.columns:
        if variant != 'ORIGINAL':
            improvement_df[f'{variant}_improvement'] = kill_rate_pivot[variant] - kill_rate_pivot['ORIGINAL']

    # Add the original kill rate for reference
    improvement_df['ORIGINAL_kill_rate'] = kill_rate_pivot['ORIGINAL']

    # Add total mutations count for context
    total_mutations_by_mutator = mutator_df[mutator_df['variant'] == 'ORIGINAL'].set_index('mutator_name')['total_mutations']
    improvement_df['total_mutations'] = total_mutations_by_mutator

    # Reorder columns to have ORIGINAL kill rate first, then improvements, then total mutations
    cols = ['ORIGINAL_kill_rate'] + [col for col in improvement_df.columns if '_improvement' in col] + ['total_mutations']
    improvement_df = improvement_df[cols]

    # Sort by the maximum improvement across all variants
    max_improvement_col = improvement_df[[col for col in improvement_df.columns if '_improvement' in col]].max(axis=1)
    improvement_df['max_improvement'] = max_improvement_col
    improvement_df = improvement_df.sort_values('max_improvement', ascending=False)

    # Drop the helper column used for sorting
    improvement_df = improvement_df.drop('max_improvement', axis=1)

    # Display the top improvements
    display(improvement_df)

    # Visualization for top 10 most improved mutators
    top_improved = improvement_df.head(10)

    plt.figure(figsize=(14, 8))

    # Set up the bar positions
    bar_width = 0.8 / len([col for col in top_improved.columns if '_improvement' in col])
    x = np.arange(len(top_improved))

    # Plot each variant's improvement
    for i, variant in enumerate([col.split('_')[0] for col in top_improved.columns if '_improvement' in col]):
        improvement_col = f'{variant}_improvement'
        plt.bar(x + i * bar_width, 
                top_improved[improvement_col], 
                width=bar_width, 
                label=f'{variant} vs ORIGINAL')

    # Add a horizontal line at y=0 to show the baseline (ORIGINAL)
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)

    plt.xlabel('Mutator')
    plt.ylabel('Kill Rate Improvement (%)')
    plt.title('Top 10 Mutators with Largest Improvements Over ORIGINAL Variant')
    plt.xticks(x + bar_width * (len([col for col in top_improved.columns if '_improvement' in col]) - 1) / 2, 
               top_improved.index, rotation=45, ha='right')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("ORIGINAL variant not found in the data. Cannot calculate improvements.")

In [ ]:
# 4. Inferential Statistics Analysis - Simplified (All Mutators)
print("\nInferential Statistics Analysis (Comparing Against Original):")

# Import required libraries
from scipy import stats
import numpy as np

stats_data = []
for _, row in mutator_df.iterrows():
    variant = row['variant']
    mutator = row['mutator_name']
    total = row['total_mutations']
    killed = row['killed_mutations']

    # Add each killed/non-killed mutation more efficiently
    stats_data.extend([{'variant': variant, 'mutator': mutator, 'killed': 1}] * killed)
    stats_data.extend([{'variant': variant, 'mutator': mutator, 'killed': 0}] * (total - killed))

stats_df = pd.DataFrame(stats_data)

# Filter for only the relevant variants
relevant_variants = ['ORIGINAL', 'NAIVE', 'IMPROVED', 'COMBINED']
stats_df = stats_df[stats_df['variant'].isin(relevant_variants)]

# Get all unique mutators
all_mutators = stats_df['mutator'].unique()

# Prepare data for comparison tests
comparison_data = []
for mutator in all_mutators:
    mutator_stats = stats_df[stats_df['mutator'] == mutator]
    for variant in stats_df['variant'].dropna().unique():
        variant_data = mutator_stats[mutator_stats['variant'] == variant]
        if len(variant_data) > 0:
            killed = variant_data['killed'].sum()
            total = len(variant_data)
            comparison_data.append({
                'mutator': mutator,
                'variant': variant,
                'kill_rate': (killed / total) * 100,
                'total_mutations': total,
                'killed': killed
            })

comparison_df = pd.DataFrame(comparison_data)

results = []
for mutator in all_mutators:
    mutator_data = comparison_df[comparison_df['mutator'] == mutator]

    # Skip if we don't have data for ORIGINAL or less than 2 variants
    if 'ORIGINAL' not in mutator_data['variant'].values or len(mutator_data) < 2:
        continue

    original_data = mutator_data[mutator_data['variant'] == 'ORIGINAL'].iloc[0]

    for variant in ['NAIVE', 'IMPROVED', 'COMBINED']:
        variant_data = mutator_data[mutator_data['variant'] == variant]
        if len(variant_data) == 0:
            continue

        variant_data = variant_data.iloc[0]
        orig_killed = original_data['killed']
        orig_total = original_data['total_mutations']
        var_killed = variant_data['killed']
        var_total = variant_data['total_mutations']

        # Fisher's exact test for comparing proportions
        contingency = [[orig_killed, orig_total - orig_killed], [var_killed, var_total - var_killed]]
        odds_ratio, p_value = stats.fisher_exact(contingency)

        # Calculate effect size (Cohen's h for proportions)
        p1 = orig_killed / orig_total
        p2 = var_killed / var_total
        h = 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

        results.append({
            'mutator': mutator,
            'variant': variant,
            'original_kill_rate': (orig_killed / orig_total) * 100,
            'variant_kill_rate': (var_killed / var_total) * 100,
            'difference': ((var_killed / var_total) - (orig_killed / orig_total)) * 100,
            'p_value': p_value,
            'significant': p_value < 0.05,
            'effect_size': abs(h),
            'effect_magnitude': 'Small' if abs(h) < 0.2 else 'Medium' if abs(h) < 0.5 else 'Large'
        })


results_df = pd.DataFrame()

# Create a summary table
if results:
    results_df = pd.DataFrame(results)

    # Sort by significance and then by effect size
    results_df = results_df.sort_values(['significant', 'effect_size'], ascending=[False, False])
else:
    print("No sufficient data for comparison.")
    
results_df


## Code coverage results pre-/post-generalization

In [ ]:
query = f"SELECT project_id, step, stage, variant, sum(instruction_missed), sum(instruction_covered), sum(branch_missed), sum(branch_covered) FROM jacoco_coverage_report WHERE project_id = {project_id} GROUP BY project_id, step, stage, variant"
df = pd.read_sql_query(query, conn)
df

## Runtime requirements per generalization variant

In [ ]:
query = f"SELECT project_id, variant, count(DISTINCT step) AS steps, count(*) AS tasks, sum(runtime) AS runtime FROM task WHERE project_id = {project_id} GROUP BY project_id, variant ORDER BY project_id"
df = pd.read_sql_query(query, conn)
df

## Runtime requirements per processing stage

In [ ]:
query = f"SELECT project_id, step, stage, variant, sum(runtime) AS runtime FROM task WHERE project_id = {project_id} GROUP BY project_id, step, stage, variant ORDER BY project_id, step"
df = pd.read_sql_query(query, conn)
df['variant'] = df['variant'].fillna('ORIGINAL')
df

In [ ]:
df['step_stage'] = df['step'].astype(str) + "-" + df['stage'].astype(str) + "-" + df['variant'].astype(str)

df.plot(kind='bar', x='step_stage', y='runtime', legend=None, figsize=(12, 6))

plt.xlabel('Processing Stage')
plt.ylabel('Runtime (in seconds)')
plt.title('Runtime per Processing Stage')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

## Causes of test failures

In [ ]:
query = f"SELECT project_id, step, stage, variant, failure_type, count(*), sum(runtime) FROM junit_test_report WHERE project_id = {project_id} GROUP BY project_id, step, stage, variant, failure_type ORDER BY step, failure_type, variant"
df = pd.read_sql_query(query, conn)
df['variant'] = df['variant'].fillna('ORIGINAL')
df['failure_type'] = df['failure_type'].fillna('')
df

## Equivalent Assertions

In [ ]:
query = f"""
SELECT 
    COUNT(*) AS total_assertions,
    SUM(CASE WHEN json_array_length(json(equivalent_assertions)) = 1 THEN 1 ELSE 0 END) AS unique_assertions,
    SUM(CASE WHEN json_array_length(json(equivalent_assertions)) > 1 THEN 1 ELSE 0 END) AS equivalent_assertions
FROM 
    assertion 
WHERE 
    project_id = {project_id} 
    AND equivalent_assertions IS NOT NULL
"""
df_unique_vs_equivalent = pd.read_sql_query(query, conn)
display(df_unique_vs_equivalent)

query = f"""
SELECT 
    json_array_length(json(equivalent_assertions)) AS equivalence_group_size
FROM 
    assertion 
WHERE 
    project_id = {project_id} 
    AND equivalent_assertions IS NOT NULL
"""
df_equivalence_groups = pd.read_sql_query(query, conn)

plt.figure(figsize=(10, 6))
counts, bins, patches = plt.hist(df_equivalence_groups['equivalence_group_size'], bins=20)

# Add count labels above each bar
for i in range(len(patches)):
    if counts[i] > 0:  # Only show labels for bars with data
        plt.text(
            bins[i] + (bins[i+1] - bins[i])/2,  # x position (center of bar)
            counts[i] + (counts.max() * 0.02),  # y position (slightly above bar)
            str(int(counts[i])),  # convert count to integer string
            ha='center',  # horizontal alignment
            va='bottom'   # vertical alignment
        )

plt.title('Distribution of Equivalence Group Sizes')
plt.xlabel('Group Size')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()